## Summary

This notebook has successfully:
<DATASET>
1. **Added BLEURT scores** to all detailed results from the <DATASET> dataset experiments
2. **Calculated Composite Quality Metrics** using both Z-score normalization and simple averaging
3. **Analyzed metric correlations** to understand relationships between different evaluation metrics
4. **Updated benchmark results** with comprehensive composite quality scores and rankings

### Key Findings:

- **BLEURT Integration**: Successfully computed BLEURT scores for all method-sample combinations
- **Composite Metrics**: Z-score normalization provides statistically sound metric aggregation
- **Method Rankings**: Composite scores offer a holistic view of method performance across all metrics
- **Correlation Analysis**: Reveals which metrics are most/least correlated, informing metric selection

### Files Generated:

- `benchmark_results_with_CI_and_composite_<DATASET>.csv`: Enhanced benchmark results with composite quality metrics
- Individual composite results for each dataset with detailed metric breakdowns

The composite quality metrics combine ROUGE-1 F1, ROUGE-2 F1, ROUGE-L F1, BERT F1, and BLEURT scores using statistically principled Z-score normalization, providing a comprehensive evaluation of summarization quality.

# Add BLEURT Scores to Benchmark Results

This notebook:
1. Reads detailed results CSV files from experiments/<DATASET>/
2. Calculates BLEURT scores using the implementation in src/evaluation_metrics.py
3. Updates the benchmark_results_with_CI_<DATASET>.csv file with BLEURT scores and confidence intervals

In [1]:
import sys
import os
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Add src directory to path
sys.path.append('src')

# Import evaluation metrics
try:
    from evaluation_metrics import EvaluationMetrics
    print("✅ Successfully imported EvaluationMetrics")
except ImportError as e:
    print(f"❌ Failed to import EvaluationMetrics: {e}")
    print("Note: Some dependencies might be missing. BLEURT scores will be set to 0.")
    EvaluationMetrics = None

✅ Successfully imported EvaluationMetrics


In [2]:
# Define paths

dataset = 'mediasum'

experiments_dir = Path(f'experiments/{dataset}')
benchmark_file = experiments_dir / f'benchmark_results_with_CI_{dataset}.csv'

print(f"Experiments directory: {experiments_dir}")
print(f"Benchmark file: {benchmark_file}")
print(f"Experiments directory exists: {experiments_dir.exists()}")
print(f"Benchmark file exists: {benchmark_file.exists()}")

Experiments directory: experiments/mediasum
Benchmark file: experiments/mediasum/benchmark_results_with_CI_mediasum.csv
Experiments directory exists: True
Benchmark file exists: True


In [3]:
# Find all detailed results CSV files
detailed_files = list(experiments_dir.glob(f'detailed_results_{dataset}_*.csv'))
print(f"Found {len(detailed_files)} detailed results files:")
for file in detailed_files:
    print(f"  - {file.name}")

Found 6 detailed results files:
  - detailed_results_mediasum_gemini.csv
  - detailed_results_mediasum_bart.csv
  - detailed_results_mediasum_distilbart.csv
  - detailed_results_mediasum_tfidfrank.csv
  - detailed_results_mediasum_textrank.csv
  - detailed_results_mediasum_GPT-5-mini.csv


In [4]:
# Read existing benchmark results
if benchmark_file.exists():
    benchmark_df = pd.read_csv(benchmark_file)
    print(f"Loaded benchmark results with {len(benchmark_df)} rows")
    print(f"Columns: {list(benchmark_df.columns)}")
    print(f"Methods: {benchmark_df['Method'].unique()}")
else:
    print("❌ Benchmark file not found!")
    benchmark_df = None

Loaded benchmark results with 6 rows
Columns: ['Method', 'Dataset', 'ROUGE-1 F1', 'ROUGE-1 F1 CI Lower', 'ROUGE-1 F1 CI Upper', 'ROUGE-1 F1 Std Error', 'ROUGE-1 F1 Sample Size', 'ROUGE-2 F1', 'ROUGE-2 F1 CI Lower', 'ROUGE-2 F1 CI Upper', 'ROUGE-2 F1 Std Error', 'ROUGE-2 F1 Sample Size', 'ROUGE-L F1', 'ROUGE-L F1 CI Lower', 'ROUGE-L F1 CI Upper', 'ROUGE-L F1 Std Error', 'ROUGE-L F1 Sample Size', 'BERT F1', 'BERT F1 CI Lower', 'BERT F1 CI Upper', 'BERT F1 Std Error', 'BERT F1 Sample Size', 'Combined Score', 'Combined Score CI Lower', 'Combined Score CI Upper', 'Combined Score Std Error', 'Combined Score Sample Size', 'Avg Time (s)', 'Avg Time (s) CI Lower', 'Avg Time (s) CI Upper', 'Avg Time (s) Std Error', 'Avg Time (s) Sample Size', 'Avg Cost ($)', 'Avg Cost ($) CI Lower', 'Avg Cost ($) CI Upper', 'Avg Cost ($) Std Error', 'Avg Cost ($) Sample Size', 'Is Best Method', 'P-value vs Best', 'Effect Size', 'Significantly Different', 'Statistical Interpretation']
Methods: ['bart' 'distilbart

In [5]:
def load_detailed_results(file_path):
    """Load detailed results and extract method name."""
    df = pd.read_csv(file_path)
    
    # Extract method name from filename
    method_name = file_path.stem.replace(f'detailed_results_{dataset}_', '')
    
    print(f"Loaded {len(df)} samples for method: {method_name}")
    
    # Check if required columns exist
    required_cols = ['reference_summary', 'generated_summary']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        print(f"⚠️  Missing columns: {missing_cols}")
        return None, method_name
    
    return df, method_name

# Load all detailed results
all_detailed_results = {}
for file_path in detailed_files:
    df, method_name = load_detailed_results(file_path)
    if df is not None:
        all_detailed_results[method_name] = df

Loaded 1000 samples for method: gemini
Loaded 1000 samples for method: bart
Loaded 1000 samples for method: distilbart
Loaded 1000 samples for method: tfidfrank
Loaded 1000 samples for method: textrank
Loaded 1000 samples for method: GPT-5-mini


In [6]:
def calculate_bleurt_scores(references, candidates, method_name):
    """Calculate BLEURT scores for a method."""
    print(f"\nCalculating BLEURT scores for {method_name}...")
    
    if EvaluationMetrics is None:
        print("⚠️  EvaluationMetrics not available, returning zeros")
        return [0.0] * len(references)
    
    try:
        evaluator = EvaluationMetrics()
        
        # Calculate BLEURT scores in batches for memory efficiency
        batch_size = 32
        all_bleurt_scores = []
        
        for i in tqdm(range(0, len(references), batch_size), desc=f"BLEURT {method_name}"):
            batch_refs = references[i:i+batch_size]
            batch_cands = candidates[i:i+batch_size]
            
            # Calculate BLEURT scores for this batch
            bleurt_result = evaluator.compute_bleurt_scores(batch_refs, batch_cands, batch_size=16)
            batch_scores = bleurt_result['bleurt_score']
            
            all_bleurt_scores.extend(batch_scores)
        
        print(f"✅ Calculated {len(all_bleurt_scores)} BLEURT scores")
        print(f"   Mean BLEURT: {np.mean(all_bleurt_scores):.4f}")
        print(f"   Std BLEURT: {np.std(all_bleurt_scores):.4f}")
        
        return all_bleurt_scores
        
    except Exception as e:
        print(f"❌ Error calculating BLEURT for {method_name}: {e}")
        return [0.0] * len(references)

In [7]:
# Calculate BLEURT scores for all methods
bleurt_results = {}

for method_name, df in all_detailed_results.items():
    print(f"\n{'='*50}")
    print(f"Processing method: {method_name}")
    print(f"{'='*50}")
    
    # Extract references and generated summaries
    references = df['reference_summary'].tolist()
    candidates = df['generated_summary'].tolist()
    
    # Clean any NaN values
    clean_pairs = [(ref, cand) for ref, cand in zip(references, candidates) 
                   if pd.notna(ref) and pd.notna(cand) and ref.strip() and cand.strip()]
    
    if len(clean_pairs) != len(references):
        print(f"⚠️  Cleaned {len(references) - len(clean_pairs)} invalid pairs")
    
    clean_refs, clean_cands = zip(*clean_pairs) if clean_pairs else ([], [])
    
    if not clean_refs:
        print(f"❌ No valid reference-candidate pairs for {method_name}")
        continue
    
    # Calculate BLEURT scores
    bleurt_scores = calculate_bleurt_scores(list(clean_refs), list(clean_cands), method_name)
    
    # Store results
    bleurt_results[method_name] = {
        'scores': bleurt_scores,
        'mean': np.mean(bleurt_scores),
        'std': np.std(bleurt_scores),
        'count': len(bleurt_scores)
    }

print(f"\n\n{'='*60}")
print("BLEURT Calculation Summary:")
print(f"{'='*60}")
for method, results in bleurt_results.items():
    print(f"{method:15} | Mean: {results['mean']:6.4f} | Std: {results['std']:6.4f} | Count: {results['count']}")


Processing method: gemini

Calculating BLEURT scores for gemini...


BLEURT gemini:   0%|                                                                        | 0/32 [00:00<?, ?it/s]

Loading BLEURT model...
BLEURT model loaded successfully on device: cpu



BLEURT gemini: 100%|███████████████████████████████████████████████████████████████| 32/32 [01:51<00:00,  3.48s/it]


✅ Calculated 1000 BLEURT scores
   Mean BLEURT: -1.1857
   Std BLEURT: 0.2859

Processing method: bart

Calculating BLEURT scores for bart...


...

✅ Calculated 1000 BLEURT scores
   Mean BLEURT: -1.2749
   Std BLEURT: 0.2778

Processing method: GPT-5-mini

Calculating BLEURT scores for GPT-5-mini...


BLEURT GPT-5-mini:   0%|                                                                    | 0/32 [00:00<?, ?it/s]

Loading BLEURT model...
BLEURT model loaded successfully on device: cpu



BLEURT GPT-5-mini: 100%|███████████████████████████████████████████████████████████| 32/32 [01:40<00:00,  3.15s/it]

✅ Calculated 1000 BLEURT scores
   Mean BLEURT: -1.1854
   Std BLEURT: 0.2863


BLEURT Calculation Summary:
gemini          | Mean: -1.1857 | Std: 0.2859 | Count: 1000
bart            | Mean: -0.9612 | Std: 0.3488 | Count: 1000
distilbart      | Mean: -0.9989 | Std: 0.3404 | Count: 1000
tfidfrank       | Mean: -1.2489 | Std: 0.2689 | Count: 1000
textrank        | Mean: -1.2749 | Std: 0.2778 | Count: 1000
GPT-5-mini      | Mean: -1.1854 | Std: 0.2863 | Count: 1000


In [8]:
pd.DataFrame(bleurt_results).T.to_csv(f'{experiments_dir}/detailed_results_{dataset}_bleurt.csv')

In [9]:
def calculate_confidence_interval(scores, confidence_level=0.95, n_bootstrap=1000):
    """Calculate bootstrap confidence interval for scores."""
    if len(scores) < 2:
        return np.mean(scores), np.mean(scores), np.mean(scores), 0.0
    
    scores = np.array(scores)
    n_samples = len(scores)
    
    # Generate bootstrap samples
    bootstrap_means = []
    np.random.seed(42)  # For reproducible results
    
    for _ in range(n_bootstrap):
        # Sample with replacement
        bootstrap_sample = np.random.choice(scores, size=n_samples, replace=True)
        bootstrap_means.append(np.mean(bootstrap_sample))
    
    bootstrap_means = np.array(bootstrap_means)
    
    # Calculate confidence interval
    alpha = 1 - confidence_level
    lower_percentile = (alpha / 2) * 100
    upper_percentile = (1 - alpha / 2) * 100
    
    mean = np.mean(scores)
    lower_bound = np.percentile(bootstrap_means, lower_percentile)
    upper_bound = np.percentile(bootstrap_means, upper_percentile)
    std_error = np.std(bootstrap_means)
    
    return mean, lower_bound, upper_bound, std_error

In [10]:
# Calculate confidence intervals for BLEURT scores
bleurt_ci_results = {}

print("Calculating confidence intervals for BLEURT scores...")
print("=" * 60)

for method_name, results in bleurt_results.items():
    scores = results['scores']
    
    mean, ci_lower, ci_upper, std_error = calculate_confidence_interval(scores)
    
    bleurt_ci_results[method_name] = {
        'mean': mean,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'std_error': std_error,
        'sample_size': len(scores)
    }
    
    print(f"{method_name:15} | Mean: {mean:6.4f} | CI: [{ci_lower:6.4f}, {ci_upper:6.4f}] | SE: {std_error:6.4f}")

print("\n✅ Confidence intervals calculated for all methods")

Calculating confidence intervals for BLEURT scores...
gemini          | Mean: -1.1857 | CI: [-1.2030, -1.1671] | SE: 0.0090
bart            | Mean: -0.9612 | CI: [-0.9851, -0.9401] | SE: 0.0112
distilbart      | Mean: -0.9989 | CI: [-1.0197, -0.9786] | SE: 0.0108
tfidfrank       | Mean: -1.2489 | CI: [-1.2655, -1.2316] | SE: 0.0086
textrank        | Mean: -1.2749 | CI: [-1.2921, -1.2574] | SE: 0.0090
GPT-5-mini      | Mean: -1.1854 | CI: [-1.2031, -1.1665] | SE: 0.0090

✅ Confidence intervals calculated for all methods


In [11]:
# Update benchmark results DataFrame with BLEURT scores
if benchmark_df is not None:
    print("\nUpdating benchmark results with BLEURT scores...")
    
    # Add BLEURT columns if they don't exist
    bleurt_columns = [
        'BLEURT Score',
        'BLEURT Score CI Lower', 
        'BLEURT Score CI Upper',
        'BLEURT Score Std Error',
        'BLEURT Score Sample Size'
    ]
    
    for col in bleurt_columns:
        if col not in benchmark_df.columns:
            benchmark_df[col] = np.nan
    
    # Update values for each method
    for method_name, ci_results in bleurt_ci_results.items():
        # Find matching row in benchmark dataframe
        mask = benchmark_df['Method'] == method_name
        
        if mask.any():
            benchmark_df.loc[mask, 'BLEURT Score'] = ci_results['mean']
            benchmark_df.loc[mask, 'BLEURT Score CI Lower'] = ci_results['ci_lower']
            benchmark_df.loc[mask, 'BLEURT Score CI Upper'] = ci_results['ci_upper']
            benchmark_df.loc[mask, 'BLEURT Score Std Error'] = ci_results['std_error']
            benchmark_df.loc[mask, 'BLEURT Score Sample Size'] = ci_results['sample_size']
            print(f"✅ Updated BLEURT scores for {method_name}")
        else:
            print(f"⚠️  Method {method_name} not found in benchmark dataframe")
    
    print("\n📊 Updated benchmark dataframe:")
    print(benchmark_df[['Method', 'BLEURT Score', 'BLEURT Score CI Lower', 'BLEURT Score CI Upper']].round(6))
else:
    print("❌ Cannot update benchmark results - dataframe not loaded")


Updating benchmark results with BLEURT scores...
✅ Updated BLEURT scores for gemini
✅ Updated BLEURT scores for bart
✅ Updated BLEURT scores for distilbart
✅ Updated BLEURT scores for tfidfrank
✅ Updated BLEURT scores for textrank
✅ Updated BLEURT scores for GPT-5-mini

📊 Updated benchmark dataframe:
       Method  BLEURT Score  BLEURT Score CI Lower  BLEURT Score CI Upper
0        bart     -0.961249              -0.985076              -0.940094
1  distilbart     -0.998925              -1.019684              -0.978648
2      gemini     -1.185747              -1.203042              -1.167127
3  GPT-5-mini     -1.185434              -1.203076              -1.166502
4    textrank     -1.274913              -1.292130              -1.257428
5   tfidfrank     -1.248869              -1.265524              -1.231587


In [12]:
# Save updated benchmark results
if benchmark_df is not None:
    output_file = experiments_dir / f'benchmark_results_with_bleurt_{dataset}.csv'
    
    try:
        benchmark_df.to_csv(output_file, index=False)
        print(f"✅ Saved updated benchmark results to: {output_file}")
        print(f"   File size: {output_file.stat().st_size / 1024:.1f} KB")
        
        # Also backup the original and replace it
        backup_file = experiments_dir / f'benchmark_results_with_CI_{dataset}_backup.csv'
        if benchmark_file.exists():
            import shutil
            shutil.copy2(benchmark_file, backup_file)
            print(f"📦 Backed up original to: {backup_file}")
            
        # Replace original with updated version
        benchmark_df.to_csv(benchmark_file, index=False)
        print(f"✅ Updated original file: {benchmark_file}")
        
    except Exception as e:
        print(f"❌ Error saving file: {e}")
else:
    print("❌ Cannot save - no dataframe to save")

✅ Saved updated benchmark results to: experiments/mediasum/benchmark_results_with_bleurt_mediasum.csv
   File size: 3.4 KB
📦 Backed up original to: experiments/mediasum/benchmark_results_with_CI_mediasum_backup.csv
✅ Updated original file: experiments/mediasum/benchmark_results_with_CI_mediasum.csv


In [13]:
# Display summary statistics
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

if benchmark_df is not None and 'BLEURT Score' in benchmark_df.columns:
    print("\n📈 BLEURT Score Summary:")
    bleurt_summary = benchmark_df[['Method', 'BLEURT Score']].sort_values('BLEURT Score', ascending=False)
    for _, row in bleurt_summary.iterrows():
        if pd.notna(row['BLEURT Score']):
            print(f"  {row['Method']:15}: {row['BLEURT Score']:7.4f}")
    
    print("\n🏆 Best method by BLEURT Score:")
    best_method = benchmark_df.loc[benchmark_df['BLEURT Score'].idxmax()]
    print(f"  {best_method['Method']}: {best_method['BLEURT Score']:.4f}")
    
    print("\n📊 BLEURT vs Other Metrics Correlation:")
    if 'Combined Score' in benchmark_df.columns:
        correlation = benchmark_df['BLEURT Score'].corr(benchmark_df['Combined Score'])
        print(f"  BLEURT vs Combined Score: {correlation:.3f}")
    
    if 'BERT F1' in benchmark_df.columns:
        correlation = benchmark_df['BLEURT Score'].corr(benchmark_df['BERT F1'])
        print(f"  BLEURT vs BERT F1: {correlation:.3f}")

print(f"\n✅ Successfully added BLEURT scores to benchmark results!")
print(f"   Total methods processed: {len(bleurt_results)}")
print(f"   Output file: {benchmark_file}")


FINAL SUMMARY

📈 BLEURT Score Summary:
  bart           : -0.9612
  distilbart     : -0.9989
  GPT-5-mini     : -1.1854
  gemini         : -1.1857
  tfidfrank      : -1.2489
  textrank       : -1.2749

🏆 Best method by BLEURT Score:
  bart: -0.9612

📊 BLEURT vs Other Metrics Correlation:
  BLEURT vs Combined Score: 0.964
  BLEURT vs BERT F1: 0.987

✅ Successfully added BLEURT scores to benchmark results!
   Total methods processed: 6
   Output file: experiments/mediasum/benchmark_results_with_CI_mediasum.csv


In [14]:
# Calculate Composite Quality Metric using Z-Score Aggregation
def calculate_composite_quality_metric(df):
    """
    Calculate Composite Quality Metric using Z-score normalization.
    Combines ROUGE-1, ROUGE-2, ROUGE-L, BERT F1, and BLEURT scores.
    """
    print("\n" + "="*80)
    print("CALCULATING COMPOSITE QUALITY METRIC")
    print("="*80)
    
    # Define the metrics to include in composite score
    metric_columns = [
        'ROUGE-1 F1',
        'ROUGE-2 F1', 
        'ROUGE-L F1',
        'BERT F1',
        'BLEURT Score'
    ]
    
    # Check which columns are available
    available_metrics = [col for col in metric_columns if col in df.columns and df[col].notna().all()]
    
    if len(available_metrics) < 2:
        print(f"⚠️  Insufficient metrics available for composite score: {available_metrics}")
        return df
    
    print(f"📊 Using metrics: {', '.join(available_metrics)}")
    
    # Calculate Z-scores for each metric
    z_scores = {}
    
    print("\n🧮 Calculating Z-scores for each metric:")
    for metric in available_metrics:
        values = df[metric].values
        mean_val = np.mean(values)
        std_val = np.std(values, ddof=1)  # Sample standard deviation
        
        if std_val == 0:
            print(f"  {metric:20}: All values identical, using zeros")
            z_scores[metric] = np.zeros_like(values)
        else:
            z_scores[metric] = (values - mean_val) / std_val
            print(f"  {metric:20}: μ={mean_val:.4f}, σ={std_val:.4f}")
    
    # Calculate composite score as average of Z-scores
    z_score_matrix = np.column_stack([z_scores[metric] for metric in available_metrics])
    composite_z_scores = np.mean(z_score_matrix, axis=1)
    
    # Add composite scores to dataframe
    df['Composite Quality Z-Score'] = composite_z_scores
    
    # Calculate traditional simple average for comparison
    simple_avg = np.mean([df[metric].values for metric in available_metrics], axis=0)
    df['Composite Quality Simple'] = simple_avg
    
    # Calculate confidence intervals for composite scores
    print("\n📈 Composite Quality Metrics Summary:")
    for method_idx, method in enumerate(df['Method']):
        z_score = composite_z_scores[method_idx]
        simple = simple_avg[method_idx]
        print(f"  {method:15}: Z-Score={z_score:7.3f}, Simple={simple:6.4f}")
    
    # Rank methods by composite score
    df['Quality Rank (Z-Score)'] = df['Composite Quality Z-Score'].rank(ascending=False, method='min').astype(int)
    df['Quality Rank (Simple)'] = df['Composite Quality Simple'].rank(ascending=False, method='min').astype(int)
    
    print(f"\n🏆 Best method by Composite Quality (Z-Score): {df.loc[df['Composite Quality Z-Score'].idxmax(), 'Method']}")
    print(f"🏆 Best method by Composite Quality (Simple): {df.loc[df['Composite Quality Simple'].idxmax(), 'Method']}")
    
    return df

# Calculate composite metrics if we have the required data
if benchmark_df is not None:
    benchmark_df = calculate_composite_quality_metric(benchmark_df)
else:
    print("❌ Cannot calculate composite metric - no benchmark data loaded")


CALCULATING COMPOSITE QUALITY METRIC
📊 Using metrics: ROUGE-1 F1, ROUGE-2 F1, ROUGE-L F1, BERT F1, BLEURT Score

🧮 Calculating Z-scores for each metric:
  ROUGE-1 F1          : μ=0.1362, σ=0.0318
  ROUGE-2 F1          : μ=0.0410, σ=0.0136
  ROUGE-L F1          : μ=0.1093, σ=0.0274
  BERT F1             : μ=0.8252, σ=0.0147
  BLEURT Score        : μ=-1.1425, σ=0.1312

📈 Composite Quality Metrics Summary:
  bart           : Z-Score=  1.338, Simple=0.0531
  distilbart     : Z-Score=  0.987, Simple=0.0395
  gemini         : Z-Score= -0.152, Simple=-0.0163
  GPT-5-mini     : Z-Score= -0.157, Simple=-0.0163
  textrank       : Z-Score= -1.002, Simple=-0.0504
  tfidfrank      : Z-Score= -1.015, Simple=-0.0465

🏆 Best method by Composite Quality (Z-Score): bart
🏆 Best method by Composite Quality (Simple): bart


In [15]:
# Optional: Display detailed comparison table
if benchmark_df is not None:
    print("\n" + "="*100)
    print("DETAILED COMPARISON TABLE")
    print("="*100)
    
    # Select key columns for comparison
    comparison_columns = [
        'Method', 
        'Composite Quality Simple',
        'ROUGE-1 F1', 
        'ROUGE-2 F1', 
        'ROUGE-L F1', 
        'BERT F1', 
        'BLEURT Score',
        # 'Combined Score',
        'Quality Rank (Simple)'
    ]
    
    available_columns = [col for col in comparison_columns if col in benchmark_df.columns]
    
    if len(available_columns) > 1:
        comparison_df = benchmark_df[available_columns].round(4)
        comparison_df = comparison_df.sort_values('Composite Quality Simple', ascending=False)
        
        print(comparison_df.to_string(index=False))
    else:
        print("⚠️  Insufficient columns for comparison table")


DETAILED COMPARISON TABLE
    Method  Composite Quality Simple  ROUGE-1 F1  ROUGE-2 F1  ROUGE-L F1  BERT F1  BLEURT Score  Quality Rank (Simple)
      bart                    0.0531      0.1771      0.0574      0.1442   0.8479       -0.9612                      1
distilbart                    0.0395      0.1684      0.0536      0.1346   0.8396       -0.9989                      2
    gemini                   -0.0163      0.1337      0.0420      0.1103   0.8184       -1.1857                      3
GPT-5-mini                   -0.0163      0.1335      0.0420      0.1100   0.8183       -1.1854                      4
 tfidfrank                   -0.0465      0.1002      0.0248      0.0774   0.8138       -1.2489                      5
  textrank                   -0.0504      0.1042      0.0259      0.0795   0.8135       -1.2749                      6
